# Broadcasting with Workflows

When a `Workflow` wraps a function that expects concrete types (e.g. `NDArray`, `float`),
but receives a `Distribution` instead, it automatically **broadcasts**: it draws samples
from the distribution, calls the function once per sample, and collects the results into
an `EmpiricalDistribution`.

This tutorial walks through the key broadcasting behaviors.

In [ ]:
import numpy as np
from numpy.typing import NDArray

from probpipe import EmpiricalDistribution, Gaussian
from probpipe.core.node import Workflow

## Basic Broadcasting

Define a simple function and wrap it in a `Workflow`. When we pass a `Gaussian`
where the function expects an `NDArray`, broadcasting kicks in.

In [ ]:
def double_it(x: NDArray) -> NDArray:
    return x * 2

w = Workflow(func=double_it, n_broadcast_samples=50)
g = Gaussian(mean=np.array([1.0, 2.0]), cov=np.eye(2))
result = w(x=g)

print(f"Result type: {type(result).__name__}")
print(f"Number of samples: {result.n}")
print(f"Dimension: {result.dim}")
print(f"First 5 samples:\n{result.samples[:5]}")

## Scalar Return Values

If the function returns a scalar, the result is still an `EmpiricalDistribution`
with `dim=1`.

In [ ]:
def compute_norm(x: NDArray) -> float:
    return float(np.linalg.norm(x))

w = Workflow(func=compute_norm, n_broadcast_samples=30)
g = Gaussian(mean=np.zeros(3), cov=np.eye(3))
result = w(x=g)

print(f"Number of samples: {result.n}")
print(f"Dimension: {result.dim}")
print(f"First 5 norms: {result.samples[:5, 0]}")

## Multiple Distribution Arguments

When multiple arguments receive distributions, each is sampled independently.

In [ ]:
def add_them(a: NDArray, b: NDArray) -> NDArray:
    return a + b

w = Workflow(func=add_them, n_broadcast_samples=30)
g1 = Gaussian(mean=np.array([1.0]), cov=np.eye(1))
g2 = Gaussian(mean=np.array([2.0]), cov=np.eye(1))
result = w(a=g1, b=g2)

print(f"Number of samples: {result.n}")
print(f"Mean (should be ~3.0): {result.samples.mean():.2f}")

## Mixed Distribution and Concrete Arguments

Only the distribution arguments are broadcast; concrete values are passed through unchanged.

In [ ]:
def scale(x: NDArray, factor: float) -> NDArray:
    return x * factor

w = Workflow(func=scale, n_broadcast_samples=20)
g = Gaussian(mean=np.array([5.0]), cov=np.eye(1))
result = w(x=g, factor=3.0)

print(f"Number of samples: {result.n}")
print(f"Mean (should be ~15.0): {result.samples.mean():.2f}")

## Controlling the Sample Count

The default number of broadcast samples is set at construction via `n_broadcast_samples`.
It can be overridden at call time.

In [ ]:
def identity(x: NDArray) -> NDArray:
    return x

w = Workflow(func=identity, n_broadcast_samples=100)
g = Gaussian(mean=np.array([0.0]), cov=np.eye(1))

result_default = w(x=g)
result_override = w(x=g, n_broadcast_samples=10)

print(f"Default: {result_default.n} samples")
print(f"Override: {result_override.n} samples")

## EmpiricalDistribution Enumeration

When an `EmpiricalDistribution` is passed and its number of samples is small
enough (at most `n_broadcast_samples`), the workflow enumerates all samples
rather than resampling. This propagates the original weights exactly.

In [ ]:
samples = np.array([[1.0], [2.0], [3.0]])
weights = np.array([0.2, 0.3, 0.5])
ed = EmpiricalDistribution(samples, weights)

w = Workflow(func=identity, n_broadcast_samples=100)
result = w(x=ed)

print(f"Input has {ed.n} samples, result has {result.n} samples (enumerated, not resampled)")
print(f"Input weights:  {ed.weights}")
print(f"Output weights: {result.weights}")
print(f"Output samples:\n{result.samples}")

## Cartesian Product of Multiple Empiricals

With two `EmpiricalDistribution` arguments, the workflow forms the cartesian
product of their samples and multiplies their weights.

In [ ]:
ed1 = EmpiricalDistribution(np.array([[1.0], [2.0]]))
ed2 = EmpiricalDistribution(np.array([[10.0], [20.0], [30.0]]))

w = Workflow(func=add_them, n_broadcast_samples=100)
result = w(a=ed1, b=ed2)

print(f"Cartesian product: {ed1.n} x {ed2.n} = {result.n} evaluations")
print(f"Result samples (a + b):\n{result.samples.ravel()}")
print(f"Weights (uniform): {result.weights}")

## Greedy Enumeration with a Budget

When the cartesian product of multiple `EmpiricalDistribution` arguments would
exceed `n_broadcast_samples`, the workflow includes as many as it can (smallest
first) and falls back to sampling the rest. Here we have three empirical
distributions whose full product would be 2 × 5 × 20 = 200, but with a budget
of 50 only the two smallest (2 × 5 = 10) are enumerated while the largest is
sampled.

In [ ]:
def sum_three(a: NDArray, b: NDArray, c: NDArray) -> NDArray:
    return a + b + c

ed_small = EmpiricalDistribution(np.array([[1.0], [2.0]]))          # n=2
ed_medium = EmpiricalDistribution(np.arange(5).reshape(-1, 1).astype(float))  # n=5
ed_large = EmpiricalDistribution(np.arange(20).reshape(-1, 1).astype(float))  # n=20

w = Workflow(func=sum_three, n_broadcast_samples=50)
result = w(a=ed_small, b=ed_medium, c=ed_large)

# Full product would be 2*5*20 = 200, but budget is 50.
# The two smallest (2*5=10) are enumerated; ed_large is sampled
# 50 // 10 = 5 samples per enumerated combo, so total = 10 * 5 = 50
print(f"Full cartesian product would be: {ed_small.n} x {ed_medium.n} x {ed_large.n} = {ed_small.n * ed_medium.n * ed_large.n}")
print(f"Actual evaluations: {result.n}")
print(f"(10 enumerated combos x 5 samples from ed_large = 50)")

## Non-Numeric Results

If the function returns a non-numeric type (e.g. strings or dicts), the result
cannot be stacked into an array, so a plain `list` is returned instead.

In [ ]:
def describe(x: NDArray) -> str:
    return f"mean={x.mean():.2f}"

w = Workflow(func=describe, n_broadcast_samples=5)
g = Gaussian(mean=np.array([1.0]), cov=np.eye(1))
result = w(x=g)

print(f"Result type: {type(result).__name__}")
print(f"Results: {result}")